#Building the Baseline
`נחבר את מנוע ה-Structured LLM Extraction (חילוץ הישויות המובנה) יחד עם מנוע ה-Geocoding הדטרמיניסטי (המרת המיקום לקואורדינטות). הקוד הבא לוקח טקסט חופשי ומבולגן של כתבה, ומפיק פלט JSON סופי ומובנה הכולל את הנקודה (Lat, Lon) על המפה`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

DATA_DIR = "/content/drive/MyDrive/From_News_to_Action/DataSets/BaseLine"

# מיפוי ידני ומדויק של מספר השורות שיש לדלג עליהן בכל קובץ (על סמך ניסוי הסריקה שלנו)
# המטרה: להגיע ישירות לשורת הנתונים הראשונה
skip_rows_mapping = {
    't01_01': 3, 't01_02': 3, 't01_03': 3, 't01_04': 3, 't01_05': 3,
    't01_06': 3, 't01_07': 3, 't01_08': 3, 't01_09': 3, 't01_10': 3,
    't01_11': 3, 't01_12': 3, 't01_13': 3, 't01_14': 4, 'sep 2026': 5
}

validated_datasets = {}

print("⚙️ מריץ טעינה סופית ומאובטחת לשמירה על כל עמודות המידע...")
print("=" * 60)

for file_prefix, skip_count in skip_rows_mapping.items():
    file = f"{file_prefix}.xlsx"
    full_path = os.path.join(DATA_DIR, file)

    try:
        # טעינה ללא הגדרת כותרת כדי למנוע מחיקת עמודות Unnamed
        df_clean = pd.read_excel(full_path, header=None, skiprows=skip_count)

        # הסרת שורות ועמודות שכולן ריקות לחלוטין (NaN)
        df_clean = df_clean.dropna(how='all').dropna(axis=1, how='all')

        # אינדוקס העמודות בצורה נקייה: עמודה 0 תהיה המשתנה המילולי, ושאר העמודות הן הנתונים
        df_clean.columns = [f"col_{i}" for i in range(df_clean.shape[1])]

        validated_datasets[file_prefix] = df_clean
        print(f"✅ {file} נטען בהצלחה! מימדים סופיים: {df_clean.shape}")

    except Exception as e:
        print(f"❌ כשל בטעינת {file}: {e}")

print(f"\n🎉 המאגרים מוכנים ומלאים בתוך validated_datasets! אפשר להמשיך הלאה.")


⚙️ מריץ טעינה סופית ומאובטחת לשמירה על כל עמודות המידע...
✅ t01_01.xlsx נטען בהצלחה! מימדים סופיים: (35, 12)
✅ t01_02.xlsx נטען בהצלחה! מימדים סופיים: (30, 12)
✅ t01_03.xlsx נטען בהצלחה! מימדים סופיים: (55, 34)
✅ t01_04.xlsx נטען בהצלחה! מימדים סופיים: (41, 15)
✅ t01_05.xlsx נטען בהצלחה! מימדים סופיים: (39, 10)
✅ t01_06.xlsx נטען בהצלחה! מימדים סופיים: (59, 20)
✅ t01_07.xlsx נטען בהצלחה! מימדים סופיים: (64, 20)
✅ t01_08.xlsx נטען בהצלחה! מימדים סופיים: (90, 20)
✅ t01_09.xlsx נטען בהצלחה! מימדים סופיים: (91, 22)
✅ t01_10.xlsx נטען בהצלחה! מימדים סופיים: (91, 20)
✅ t01_11.xlsx נטען בהצלחה! מימדים סופיים: (35, 20)
✅ t01_12.xlsx נטען בהצלחה! מימדים סופיים: (35, 20)
✅ t01_13.xlsx נטען בהצלחה! מימדים סופיים: (172, 16)
✅ t01_14.xlsx נטען בהצלחה! מימדים סופיים: (37, 18)
✅ sep 2026.xlsx נטען בהצלחה! מימדים סופיים: (49, 12)

🎉 המאגרים מוכנים ומלאים בתוך validated_datasets! אפשר להמשיך הלאה.


In [ ]:
!pip install langchain-openai pydantic requests


#להלן שלד ה-Pipeline המלא שמחבר את המודל (לדוגמה GEMINI 3.8 Flash) ישירות ל-OpenStreetMap API:

In [ ]:
import os
import requests
import time
from typing import List, Tuple, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 1. הגדרת מבנה הנתונים הקשיח באמצעות Pydantic
class TrafficAccidentExtract(BaseModel):
    accident_severity: str = Field(description="חומרת התאונה: קלה, בינונית, קשה, קטלנית, או 'לא מצוין'")
    vehicle_types: List[str] = Field(description="רשימת סוגי הרכבים המעורבים (למשל: אוטובוס, אופנוע, רכב פרטי)")
    human_factor: str = Field(description="הגורם האנושי המוזכר: הסחת דעת, מהירות מופרזת, אי ציות לרמזור, או 'לא מצוין'")
    infrastructure_factor: str = Field(description="מפגעי תשתית: בור בכביש, חוסר תאורה, כביש רטוב/חלק, או 'אין'")
    verbal_location: str = Field(description="תיאור המיקום המילולי המדויק ביותר לחילוץ (למשל: 'כביש 4 ליד מחלף גהה לכיוון דרום')")

# 2. פונקציית ה-Geocoding הדטרמיניסטית (OpenStreetMap Nominatim)
def geocode_location(location_name: str) -> Tuple[Optional[float], Optional[float]]:
    url = "https://openstreetmap.org"
    params = {'q': location_name, 'format': 'json', 'countrycodes': 'il', 'limit': 1}
    headers = {'User-Agent': 'From_News_to_Action_Baseline_v1.0 (contact: final_project@ai_course.com)'}

    try:
        response = requests.get(url, params=params, headers=headers)
        time.sleep(1.0)  # השהייה חובה של שנייה לפי תנאי השימוש של OSM
        if response.status_code == 200 and len(response.json()) > 0:
            result = response.json()[0]
            return float(result['lat']), float(result['lon'])
    except Exception as e:
        print(f"⚠️ שגיאה ב-Geocoding עבור '{location_name}': {e}")
    return None, None

# 3. פונקציית ה-Pipeline המרכזית (End-to-End)
def run_baseline_pipeline(raw_text: str, openai_api_key: str) -> dict:
    print("🧠 שלב 1: מריץ חילוץ ישויות מובנה באמצעות LLM...")

    # אתחול המודל עם אכיפת מבנה הנתונים (Structured Output)
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=openai_api_key)
    structured_llm = llm.with_structured_output(TrafficAccidentExtract)

    # ביצוע החילוץ מהטקסט
    extracted_data = structured_llm.invoke(raw_text)

    # המרת אובייקט ה-Pydantic לדיקשנרי רגיל
    output_json = extracted_data.model_dump()
    print(f"   • חולץ מיקום מילולי: '{output_json['verbal_location']}'")

    print("🌍 שלב 2: שולח את המיקום המילולי למנוע ה-Geocoding...")
    lat, lon = geocode_location(output_json['verbal_location'])

    # העשרת ה-JSON בנקודת הציון (Point) שנתקבלה על המפה
    output_json['location_gps'] = {
        'latitude': lat,
        'longitude': lon
    }

    print("🎯 ה-Pipeline הסתיים בהצלחה!")
    return output_json


#Checking the POC with feeding values

In [ ]:
!pip install -q google-genai pydantic requests


In [ ]:
import os
import requests
import time
import urllib.parse
import json
from typing import List, Tuple, Optional
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata

# ==========================================
# 1. הגדרת מבנה הנתונים הקשיח באמצעות Pydantic
# ==========================================
class TrafficAccidentExtract(BaseModel):
    accident_severity: str = Field(description="חומרת התאונה: קלה, בינונית, קשה, קטלנית, או 'לא מצוין'")
    vehicle_types: List[str] = Field(description="רשימת סוגי הרכבים המעורבים (למשל: אוטובוס, אופנוע, רכב פרטי)")
    human_factor: str = Field(description="הגורם האנושי המוזכר: הסחת דעת, מהירות מופרזת, אי ציות לרמזור, או 'לא מצוין'")
    infrastructure_factor: str = Field(description="מפגעי תשתית: בור בכביש, חוסר תאורה, כביש רטוב/חלק, או 'אין'")
    verbal_location: str = Field(description="תיאור המיקום המילולי המדויק ביותר לחילוץ (למשל: 'כביש 4 ליד מחלף גהה לכיוון דרום')")

# ==========================================
# 2. התיקון הגיאוגרפי הסופי: הוספת עוגן מדינה דטרמיניסטי
# ==========================================
def query_nominatim(q_text: str) -> Tuple[Optional[float], Optional[float]]:
    url = "https://openstreetmap.org"
    params = {
        'q': q_text,
        'format': 'json',
        'countrycodes': 'il',
        'limit': 1
    }
    headers = {
        'User-Agent': 'From_News_to_Action_Baseline_v2.3 (contact: final_project@ai_course.com)'
    }
    try:
        response = requests.get(url, params=params, headers=headers)
        time.sleep(1.0) # השהיית חובה של שנייה לפי חוקי OSM
        if response.status_code == 200 and len(response.json()) > 0:
            result = response.json()
            return float(result['lat']), float(result['lon'])
    except Exception as e:
        print(f"⚠️ שגיאה בפנייה למפה עבור '{q_text}': {e}")
    return None, None

def geocode_location_advanced(location_name: str) -> Tuple[Optional[float], Optional[float]]:
    # שלב א': ניקוי מילים מפריעות כלליות שמשבשות מנועי חיפוש
    clean_location = location_name
    words_to_remove = ["לכיוון דרום", "לכיוון צפון", "לכיוון מזרח", "לכיוון מערב", "סמוך ל", "ליד", "בסמוך ל", "מבזק חם מחמל:"]
    for word in words_to_remove:
        clean_location = clean_location.replace(word, "")
    clean_location = clean_location.strip()

    # ניסיון 1: חיפוש העוגן הממוקד עם תוספת המדינה "ישראל" כדי להבטיח זיהוי
    # למשל: "כביש 4 מחלף גהה, ישראל"
    query_1 = f"{clean_location}, ישראל"
    print(f"   🌐 ניסיון גיאוקודינג 1: מחפש '{query_1}'...")
    lat, lon = query_nominatim(query_1)
    if lat and lon:
        return lat, lon

    # ניסיון 2 (Fallback): ניקוי הקידומת של מספר הכביש וחיפוש העוגן המרכזי בלבד
    # למשל: "מחלף גהה, ישראל"
    if "כביש" in clean_location:
        parts = clean_location.split()
        fallback_query = " ".join([p for i, p in enumerate(parts) if p != "כביש" and (i == 0 or parts[i-1] != "כביש")])
        query_2 = f"{fallback_query.strip()}, ישראל"

        print(f"   🔄 ניסיון גיאוקודינג 2 (Fallback): מחפש את העוגן החלק '{query_2}'...")
        lat, lon = query_nominatim(query_2)
        if lat and lon:
            return lat, lon

    return None, None

# ==========================================
# 3. פונקציית ה-Pipeline המרכזית (End-to-End)
# ==========================================
def run_baseline_pipeline_genai_sdk(raw_text: str, google_api_key: str) -> dict:
    client = genai.Client(api_key=google_api_key)

    prompt = f"""
    אתה אנליסט נתונים מומחה של הרשות הלאומית לבטיחות בדרכים.
    נתח את טקסט דיווח התאונה הבא וחלץ מתוכו את הפרטים המדויקים על פי מבנה הנתונים הנדרש בלבד.

    הטקסט לניתוח:
    {raw_text}
    """

    max_retries = 3
    retry_delay = 4.0
    response = None

    for attempt in range(max_retries):
        try:
            print(f"🧠 שלב 1 (ניסיון {attempt + 1}/{max_retries}): מריץ חילוץ ישויות מובנה...")
            response = client.models.generate_content(
                model='gemini-3.8-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=TrafficAccidentExtract,
                    temperature=0.0
                ),
            )
            break
        except APIError as e:
            if e.code == 503 or "demand" in str(e).lower():
                if attempt < max_retries - 1:
                    print(f"⚠️ השרת בעומס זמני. ממתין {retry_delay} שניות...")
                    time.sleep(retry_delay)
                    retry_delay *= 1.5
                    continue
            raise e

    output_json = json.loads(response.text)
    print(f"   • חולץ מיקום מילולי: '{output_json['verbal_location']}'")

    print("🌍 שלב 2: מעביר את המיקום למערכת הגיאוקודינג הרב-שלבית...")
    lat, lon = geocode_location_advanced(output_json['verbal_location'])

    output_json['location_gps'] = {
        'latitude': lat,
        'longitude': lon
    }

    print("🎯 ה-Pipeline הסתיים בהצלחה!")
    return output_json

# ==========================================
# 4. בלוק ההרצה המרכזי
# ==========================================
try:
    GEMINI_KEY = userdata.get('News_to_Action')
    print("🔑 המפתח 'News_to_Action' נמשך בהצלחה מתוך ה-Secrets!")
except Exception:
    GEMINI_KEY = None

sample_news_text = """
מבזק חם מחמל: תאונה קשה התרחשה לפני זמן קצר בכביש 4 סמוך למחלף גהה לכיוון דרום.
מדווח על התנגשות חזיתית בין אוטובוס של אגד לבין שני רכבים פרטיים.
צוותי מד"א שהגיעו למקום מעניקים טיפול רפואי לשלושה פצועים בדרגות חומרה שונות.
בוחני תנועה מעריכים כי הגורם לתאונה הוא כביש רטוב וחלק בעקבות הגשם הראשון,
יחד עם חוסר תאורה מוחלט בקטע הכביש המדובר. הנהגים מתבקשים לנסוע בדרכים חלופיות.
"""

if GEMINI_KEY:
    try:
        final_result = run_baseline_pipeline_genai_sdk(sample_news_text, GEMINI_KEY)
        print("\n🎯 פלט ה-Baseline המובנה והנקי המלא:")
        print(json.dumps(final_result, indent=4, ensure_ascii=False))
    except Exception as e:
        print(f"\n❌ הרצת הצינור נכשלה: {e}")


🔑 המפתח 'News_to_Action' נמשך בהצלחה מתוך ה-Secrets!
🧠 שלב 1 (ניסיון 1/3): מריץ חילוץ ישויות מובנה...
   • חולץ מיקום מילולי: 'כביש 4 סמוך למחלף גהה לכיוון דרום'
🌍 שלב 2: מעביר את המיקום למערכת הגיאוקודינג הרב-שלבית...
   🌐 ניסיון גיאוקודינג 1: מחפש 'כביש 4 מחלף גהה, ישראל'...
   🔄 ניסיון גיאוקודינג 2 (Fallback): מחפש את העוגן החלק 'מחלף גהה, ישראל'...
🎯 ה-Pipeline הסתיים בהצלחה!

🎯 פלט ה-Baseline המובנה והנקי המלא:
{
    "accident_severity": "קשה",
    "vehicle_types": [
        "אוטובוס",
        "רכב פרטי"
    ],
    "human_factor": "לא מצוין",
    "infrastructure_factor": "כביש רטוב/חלק, חוסר תאורה",
    "verbal_location": "כביש 4 סמוך למחלף גהה לכיוון דרום",
    "location_gps": {
        "latitude": null,
        "longitude": null
    }
}


#The deterministic engine is having some difficuties to locate GEOsiting in Roads with OpenStreetMaps so we have altered the to Google maps

In [ ]:
import os
import requests
import time
import urllib.parse
import json
from typing import List, Tuple, Optional
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata

# ==========================================
# 1. הגדרת מבנה הנתונים הקשיח באמצעות Pydantic
# ==========================================
class TrafficAccidentExtract(BaseModel):
    accident_severity: str = Field(description="חומרת התאונה: קלה, בינונית, קשה, קטלנית, או 'לא מצוין'")
    vehicle_types: List[str] = Field(description="רשימת סוגי הרכבים המעורבים (למשל: אוטובוס, אופנוע, רכב פרטי)")
    human_factor: str = Field(description="הגורם האנושי המוזכר: הסחת דעת, מהירות מופרזת, אי ציות לרמזור, או 'לא מצוין'")
    infrastructure_factor: str = Field(description="מפגעי תשתית: בור בכטיש, חוסר תאורה, כביש רטוב/חלק, או 'אין'")
    verbal_location: str = Field(description="תיאור המיקום המילולי המדויק ביותר לחילוץ (למשל: 'כביש 4 ליד מחלף גהה לכיוון דרום')")

# ==========================================
# 2. מנוע גיאוקודינג דטרמיניסטי פשוט (מתוקן)
# ==========================================
def query_nominatim(q_text: str) -> Tuple[Optional[float], Optional[float]]:
    # תיקון קריטי: שינוי ה-URL ל-Endpoint הרשמי של Nominatim לצורך קבלת פלט JSON תקין
    url = "https://openstreetmap.org"
    params = {'q': q_text, 'format': 'json', 'countrycodes': 'il', 'limit': 1}
    headers = {'User-Agent': 'From_News_to_Action_Baseline_v2.5 (contact: final_project@ai_course.com)'}
    try:
        response = requests.get(url, params=params, headers=headers)
        time.sleep(1.0) # שמירה על חוקי ה-Rate Limit של OSM
        if response.status_code == 200 and len(response.json()) > 0:
            result = response.json()[0] # שליפת האיבר הראשון ברשימת התוצאות
            return float(result['lat']), float(result['lon'])
    except Exception as e:
        print(f"      [DEBUG API ERROR]: {e}")
        pass
    return None, None

# פונקציית עזר מוגנת עומסים לקריאה מול מודל השפה
def safe_generate_content(client: genai.Client, model_name: str, contents: str, config=None) -> str:
    max_retries = 5
    retry_delay = 4.0
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=model_name, contents=contents, config=config)
            return response.text
        except APIError as e:
            if e.code in [429, 503] or "demand" in str(e).lower() or "temporary" in str(e).lower():
                if attempt < max_retries - 1:
                    print(f"   ⚠️ השרת חווה עומס זמני (ניסיון {attempt + 1}/{max_retries}). ממתין {retry_delay} שניות...")
                    time.sleep(retry_delay)
                    retry_delay *= 2.0
                    continue
            raise e
    raise RuntimeError("כשל בפנייה למודל עקב עומס בשרתי גוגל.")

def geocode_location_advanced_v3(location_name: str, client: genai.Client) -> Tuple[Optional[float], Optional[float]]:
    clean_location = location_name
    # ניקוי אגרסיבי של רעש מילולי שמשגע מנועי מפות קשיחים
    for word in ["לכיוון דרום", "לכיוון צפון", "לכיוון מזרח", "לכיוון מערב",
                 "סמוך ל", "ליד", "בסמוך ל", "כביש 4", "כביש 1", "כביש 6", "כביש"]:
        clean_location = clean_location.replace(word, "")
    clean_location = clean_location.strip()

    # ניסיון 1: חיפוש ישיר של השם הנקי בעברית (למשל: "מחלף גהה")
    print(f"   🌐 ניסיון גיאוקודינג 1: מחפש '{clean_location}, ישראל'...")
    lat, lon = query_nominatim(f"{clean_location}, ישראל")
    if lat and lon:
        print("   ✅ מיקום נמצא בניסיון 1!")
        return lat, lon

    # ניסיון 2: התאמת מונח עברי (מחלף -> צומת)
    junc_location = clean_location.replace("מחלף", "צומת")
    print(f"   🌐 ניסיון גיאוקודינג 2 (התאמת מונח עברי): מחפש '{junc_location}, ישראל'...")
    lat, lon = query_nominatim(f"{junc_location}, ישראל")
    if lat and lon:
        print("   ✅ מיקום נמצא בניסיון 2!")
        return lat, lon

    # ניסיון 3 (Fallback): זיקוק השם הרשמי המדויק ביותר ב-OSM (ללא מספרי דרכים)
    print("   🔄 ניסיון גיאוקודינג 3 (DCAI Fallback): מבקש מהמודל את שם הישות הנוקשה בלבד...")
    translation_prompt = f"""
    You are a GIS data cleaner. Extract ONLY the core name of the junction or interchange from the text.
    Remove any road numbers, highway prefixes, or directions.
    Convert it to its official English mapping name.
    Examples:
    - 'מחלף גהה' -> 'Geha Interchange'
    - 'צומת גילת' -> 'Gilat Junction'

    Output ONLY the final string (2-3 words), with NO punctuation and NO explanation.
    Text: {clean_location}
    """
    try:
        translated_text = safe_generate_content(client, 'gemini-3.8-flash', translation_prompt)
        english_location = translated_text.strip().replace("'", "").replace('"', "")

        # ננסה לחפש את השם באנגלית, ואם נכשל ננסה להוסיף את העיר תל אביב/בני ברק כעוגן רדיוס
        for query_text in [f"{english_location}, Israel", f"{english_location}, Bnei Brak, Israel"]:
            print(f"   🌐 מחפש במפה את השם הממוקד: '{query_text}'...")
            lat, lon = query_nominatim(query_text)
            if lat and lon:
                print(f"   ✅ מיקום נמצא בניסיון 3 (Fallback) תחת השם: {query_text}!")
                return lat, lon
    except Exception as e:
        print(f"   ⚠️ כשל בשלב ה-Fallback הגיאוגרפי: {e}")

    return None, None

# ==========================================
# 3. פונקציית ה-Pipeline המרכזית (End-to-End)
# ==========================================
def run_baseline_pipeline_genai_sdk(raw_text: str, google_api_key: str) -> dict:
    client = genai.Client(api_key=google_api_key)

    prompt = f"""
    אתה אנליסט נתונים מומחה של הרשות הלאומית לבטיחות בדרכים.
    נתח את טקסט דיווח התאונה הבא וחלץ מתוכו את הפרטים המדויקים על פי מבנה הנתונים הנדרש בלבד.

    הטקסט לניתוח:
    {raw_text}
    """

    print("🧠 שלב 1: מריץ חילוץ ישויות מובנה מוגן עומסים במודל gemini-3.8-flash...")
    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=TrafficAccidentExtract,
        temperature=0.0
    )

    raw_response = safe_generate_content(client, 'gemini-3.8-flash', prompt, config)
    output_json = json.loads(raw_response)
    print(f"   • חולץ מיקום מילולי: '{output_json['verbal_location']}'")

    print("🌍 שלב 2: מעביר את המיקום למערכת הגיאוקודינג המעודכנת...")
    lat, lon = geocode_location_advanced_v3(output_json['verbal_location'], client)

    output_json['location_gps'] = {
        'latitude': lat,
        'longitude': lon
    }

    print("🎯 ה-Pipeline הסתיים בהצלחה!")
    return output_json

# ==========================================
# 4. בלוק ההרצה המרכזי
# ==========================================
try:
    GEMINI_KEY = userdata.get('News_to_Action')
    print("🔑 המפתח 'News_to_Action' נמשך בהצלחה מתוך ה-Secrets!")
except Exception:
    GEMINI_KEY = None

sample_news_text = """
מבזק חם מחמל: תאונה קשה התרחשה לפני זמן קצר בכביש 4 סמוך למחלף גהה לכיוון דרום.
מדווח על התנגשות חזיתית בין אוטובוס של אגד לבין שני רכבים פרטיים.
צוותי מד"א שהגיעו למקום מעניקים טיפול רפואי לשלושה פצועים בדרגות חומרה שונות.
בוחני תנועה מעריכים כי הגורם לתאונה הוא כביש רטוב וחלק בעקבות הגשם הראשון,
יחד עם חוסר תאורה מוחלט בקטע הכביש המדובר. הנהגים מתבקשים לנסוע בדרכים חלופיות.
"""

if GEMINI_KEY:
    try:
        final_result = run_baseline_pipeline_genai_sdk(sample_news_text, GEMINI_KEY)
        print("\n🎯 פלט ה-Baseline המובנה והנקי המלא:")
        print(json.dumps(final_result, indent=4, ensure_ascii=False))
    except Exception as e:
        print(f"\n❌ הרצת הצינור נכשלה: {e}")
else:
    print("\n❌ שגיאה: לא נמצא מפתח API תחת השם 'News_to_Action' ב-Secrets של Colab.")


🔑 המפתח 'News_to_Action' נמשך בהצלחה מתוך ה-Secrets!
🧠 שלב 1: מריץ חילוץ ישויות מובנה מוגן עומסים במודל gemini-3.8-flash...
   • חולץ מיקום מילולי: 'כביש 4 סמוך למחלף גהה לכיוון דרום'
🌍 שלב 2: מעביר את המיקום למערכת הגיאוקודינג המעודכנת...
   🌐 ניסיון גיאוקודינג 1: מחפש 'מחלף גהה, ישראל'...
   🌐 ניסיון גיאוקודינג 2 (התאמת מונח עברי): מחפש 'צומת גהה, ישראל'...
   🔄 ניסיון גיאוקודינג 3 (DCAI Fallback): מבקש מהמודל את שם הישות הנוקשה בלבד...
   🌐 מחפש במפה את השם הממוקד: 'Geha Interchange, Israel'...
   🌐 מחפש במפה את השם הממוקד: 'Geha Interchange, Bnei Brak, Israel'...
🎯 ה-Pipeline הסתיים בהצלחה!

🎯 פלט ה-Baseline המובנה והנקי המלא:
{
    "accident_severity": "קשה",
    "vehicle_types": [
        "אוטובוס",
        "רכב פרטי"
    ],
    "human_factor": "לא מצוין",
    "infrastructure_factor": "כביש רטוב/חלק, חוסר תאורה",
    "verbal_location": "כביש 4 סמוך למחלף גהה לכיוון דרום",
    "location_gps": {
        "latitude": null,
        "longitude": null
    }
}


In [ ]:
!pip install -q googlemaps


  Preparing metadata (setup.py) ... done


In [ ]:
!pip install -q googlemaps

import googlemaps
from google.colab import userdata

# 1. הגדרת שם המפתח כפי שהוא רשום ב-Secrets שלך
# על פי היסטוריית הפרויקט שלך, השם שהגדרת הוא 'Google_Map'
SECRET_NAME = 'Google_Map'

print("🔑 שלב 1: מנסה למשוך את המפתח מתוך ה-Secrets של Colab...")
try:
    MAPS_KEY = userdata.get(SECRET_NAME)
    if MAPS_KEY:
        print(f"   ✅ המפתח '{SECRET_NAME}' נמשך בהצלחה מה-Secrets!")
        # הדפסת הצצה קטנה למפתח כדי לוודא שהוא לא ריק (בלי לחשוף את כולו)
        print(f"   • תחילת המפתח: {MAPS_KEY[:6]}...{MAPS_KEY[-4:]}")
    else:
        print(f"   ❌ המפתח '{SECRET_NAME}' נמצא, אך הוא ריק לחלוטין.")
        MAPS_KEY = None
except Exception as e:
    print(f"   ❌ כשל במשיכת המפתח. ודאו שהשם '{SECRET_NAME}' רשום במדויק ב-Secrets ושהרשאת הגישה למחברת (Notebook access) מופעלת.")
    print(f"   • שגיאה: {e}")
    MAPS_KEY = None

# 2. בדיקת תקינות מול שרתי Google Maps
if MAPS_KEY:
    print("\n🌐 שלב 2: שולח בקשת בדיקה ל-Google Maps API כדי לוודא שהמפתח תקין ופעיל...")
    try:
        # אתחול הלקוח של גוגל
        gmaps = googlemaps.Client(key=MAPS_KEY)

        # הרצת גיאוקודינג פשוט על "מחלף גהה"
        test_query = "מחלף גהה, ישראל"
        geocode_result = gmaps.geocode(test_query)

        if geocode_result:
            print("   ✅ ה-API KEY תקין לחלוטין ופעיל!")
            # שליפת המידע שחזר מגוגל
            location = geocode_result[0]['geometry']['location']
            formatted_address = geocode_result[0]['formatted_address']
            print(f"   • כתובת רשמית שגוגל מצא: {formatted_address}")
            print(f"   • קואורדינטות: Latitude: {location['lat']}, Longitude: {location['lng']}")
        else:
            print("   ⚠️ פנייה ל-API הצליחה, אך גוגל מפות לא מצא תוצאות עבור 'מחלף גהה'.")

    except Exception as e:
        print("   ❌ ה-API KEY נמשך, אך הוא נדחה על ידי השרתים של גוגל!")
        print("   • מומלץ לבדוק ב-Google Cloud Console שהמפתח לא פג תוקף, שחבילת ה-Geocoding API מופעלת (Enabled) ושאין מגבלות חסימה על המפתח.")
        print(f"   • פרטי השגיאה מגוגל: {e}")
else:
    print("\n❌ לא ניתן לבצע את בדיקת השרת מכיוון שהמפתח לא נמשך בהצלחה.")


🔑 שלב 1: מנסה למשוך את המפתח מתוך ה-Secrets של Colab...
   ✅ המפתח 'Google_Map' נמשך בהצלחה מה-Secrets!
   • תחילת המפתח: AIzaSy...L20s

🌐 שלב 2: שולח בקשת בדיקה ל-Google Maps API כדי לוודא שהמפתח תקין ופעיל...
   ✅ ה-API KEY תקין לחלוטין ופעיל!
   • כתובת רשמית שגוגל מצא: Geha Interchange, Petah Tikva
   • קואורדינטות: Latitude: 32.0923659, Longitude: 34.8459393


In [ ]:
import os
import requests
import time
import json
import googlemaps
from typing import List, Tuple, Optional
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata

# ==========================================
# 1. הגדרת מבנה הנתונים הקשיח באמצעות Pydantic
# ==========================================
class TrafficAccidentExtract(BaseModel):
    accident_severity: str = Field(description="חומרת התאונה: קלה, בינונית, קשה, קטלנית, או 'לא מצוין'")
    vehicle_types: List[str] = Field(description="רשימת סוגי הרכבים המעורבים (למשל: אוטובוס, אופנוע, רכב פרטי)")
    human_factor: str = Field(description="הגורם האנושי המוזכר: הסחת דעת, מהירות מופרזת, אי ציות לרמזור, או 'לא מצוין'")
    infrastructure_factor: str = Field(description="מפגעי תשתית: בור בכביש, חוסר תאורה, כביש רטוב/חלק, או 'אין'")
    verbal_location: str = Field(description="תיאור המיקום המילולי המדויק ביותר לחילוץ (למשל: 'כביש 4 ליד מחלף גהה לכיוון דרום')")

# ==========================================
# 2. מנוע גיאוקודינג של Google Maps
# ==========================================
def geocode_with_google(verbal_location: str, gmaps_client: googlemaps.Client) -> Tuple[Optional[float], Optional[float]]:
    try:
        # פנייה רשמית לגוגל מפות עם עוגן מדינת ישראל
        geocode_result = gmaps_client.geocode(f"{verbal_location}, ישראל")

        if geocode_result:
            location = geocode_result[0]['geometry']['location']
            formatted_address = geocode_result[0]['formatted_address']
            print(f"   ✅ גוגל מפות מצא את המיקום: '{formatted_address}'")
            return float(location['lat']), float(location['lng'])
        else:
            print("   ❌ גוגל מפות לא מצא תוצאות עבור המיקום הזה.")
    except Exception as e:
        print(f"   ⚠️ שגיאה בפנייה ל-Google Maps API: {e}")
    return None, None

# פונקציית עזר מוגנת עומסים לקריאה מול מודל השפה
def safe_generate_content(client: genai.Client, model_name: str, contents: str, config=None) -> str:
    max_retries = 5
    retry_delay = 4.0
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=model_name, contents=contents, config=config)
            return response.text
        except APIError as e:
            if e.code in [429, 503] or "demand" in str(e).lower() or "temporary" in str(e).lower():
                if attempt < max_retries - 1:
                    print(f"   ⚠️ השרת חווה עומס זמני (ניסיון {attempt + 1}/{max_retries}). ממתין {retry_delay} שניות...")
                    time.sleep(retry_delay)
                    retry_delay *= 2.0
                    continue
            raise e
    raise RuntimeError("כשל בפנייה למודל עקב עומס בשרתי גוגל.")

# ==========================================
# 3. פונקציית ה-Pipeline המרכזית (End-to-End)
# ==========================================
def run_baseline_pipeline_with_google(raw_text: str, gemini_key: str, google_maps_key: str) -> dict:
    # אתחול הלקוחות של גוגל (מודל השפה והמפות)
    gemini_client = genai.Client(api_key=gemini_key)
    gmaps_client = googlemaps.Client(key=google_maps_key)

    prompt = f"""
    אתה אנליסט נתונים מומחה של הרשות הלאומית לבטיחות בדרכים.
    נתח את טקסט דיווח התאונה הבא וחלץ מתוכו את הפרטים המדויקים על פי מבנה הנתונים הנדרש בלבד.

    הטקסט לניתוח:
    {raw_text}
    """

    print("🧠 שלב 1: מריץ חילוץ ישויות מובנה (Structured Output) במודל gemini-3.8-flash...")
    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=TrafficAccidentExtract,
        temperature=0.0
    )

    raw_response = safe_generate_content(gemini_client, 'gemini-3.8-flash', prompt, config)
    output_json = json.loads(raw_response)
    print(f"   • חולץ מיקום מילולי מהטקסט: '{output_json['verbal_location']}'")

    print("🌍 שלב 2: שולח את המיקום המילולי ל-Google Maps Geocoding API...")
    lat, lon = geocode_with_google(output_json['verbal_location'], gmaps_client)

    # הזרקת הקואורדינטות לתוך ה-JSON הסופי
    output_json['location_gps'] = {
        'latitude': lat,
        'longitude': lon
    }

    print("🎯 ה-Pipeline הסתיים בהצלחה!")
    return output_json

# ==========================================
# 4. בלוק ההרצה המרכזי
# ==========================================
# משיכת שני המפתחות מה-Secrets שלכן
try:
    GEMINI_KEY = userdata.get('News_to_Action')
    MAPS_KEY = userdata.get('Google_Map')
    print("🔑 שני המפתחות ('News_to_Action' ו-'Google_Map') נמשכו בהצלחה!")
except Exception as e:
    print(f"❌ שגיאה במשיכת המפתחות: {e}")
    GEMINI_KEY, MAPS_KEY = None, None

sample_news_text = """
מבזק חם מחמל: תאונה קשה התרחשה לפני זמן קצר בכביש 4 סמוך למחלף גהה לכיוון דרום.
מדווח על התנגשות חזיתית בין אוטובוס של אגד לבין שני רכבים פרטיים.
צוותי מד"א שהגיעו למקום מעניקים טיפול רפואי לשלושה פצועים בדרגות חומרה שונות.
בוחני תנועה מעריכים כי הגורם לתאונה הוא כביש רטוב וחלק בעקבות הגשם הראשון,
יחד עם חוסר תאורה מוחלט בקטע הכביש המדובר. הנהגים מתבקשים לנסוע בדרכים חלופיות.
"""

if GEMINI_KEY and MAPS_KEY:
    try:
        final_result = run_baseline_pipeline_with_google(sample_news_text, GEMINI_KEY, MAPS_KEY)
        print("\n🎯 פלט ה-Baseline המובנה, הנקי והממופה המלא:")
        print(json.dumps(final_result, indent=4, ensure_ascii=False))
    except Exception as e:
        print(f"\n❌ הרצת הצינור נכשלה: {e}")


🔑 שני המפתחות ('News_to_Action' ו-'Google_Map') נמשכו בהצלחה!
🧠 שלב 1: מריץ חילוץ ישויות מובנה (Structured Output) במודל gemini-3.8-flash...
   • חולץ מיקום מילולי מהטקסט: 'כביש 4 סמוך למחלף גהה לכיוון דרום'
🌍 שלב 2: שולח את המיקום המילולי ל-Google Maps Geocoding API...
   ✅ גוגל מפות מצא את המיקום: 'Geha Interchange, Petah Tikva'
🎯 ה-Pipeline הסתיים בהצלחה!

🎯 פלט ה-Baseline המובנה, הנקי והממופה המלא:
{
    "accident_severity": "קשה",
    "vehicle_types": [
        "אוטובוס",
        "רכב פרטי"
    ],
    "human_factor": "לא מצוין",
    "infrastructure_factor": "כביש רטוב/חלק, חוסר תאורה",
    "verbal_location": "כביש 4 סמוך למחלף גהה לכיוון דרום",
    "location_gps": {
        "latitude": 32.0923659,
        "longitude": 34.8459393
    }
}
